### **Кросс-валидация и регуляризация.**
* _будем замерять качество, используя кросс-валидацию_
* _на изначальных данных_
* _на данных с дополнительными линейно зависимыми признаками_
* _на данных с доп признаками, но используя кросс-валидацию_

In [1]:
import pandas as pd
import numpy as np

_Считаем данные. Сделав id поездки индексом строк._

In [2]:
data = pd.read_csv('../../5. Обобщающая способность/5 processed_data.csv', index_col='id')

data.head()

,vendor_id,passenger_count,store_and_fwd_flag,trip_duration,distance_km
id,,,,,
id2875421,1,930.399753,0,455.0,1.500479
id2377394,0,930.399753,0,663.0,1.807119
id3858529,1,930.399753,0,2124.0,6.392080
id3504673,1,930.399753,0,429.0,1.487155
id2181028,1,930.399753,0,435.0,1.189925


_Определимся с признаками и таргетами._

In [3]:
x = data.drop('trip_duration', axis=1)
y = data['trip_duration']

_**Построим Линейную регрессию и замерим качество на кросс валидации.**_

_Проведем кросс-валидацию вручную._

In [4]:
from sklearn.model_selection import KFold
#создадим правило для разделения данных
kf = KFold(n_splits=4, shuffle=True, random_state=33)

from sklearn.linear_model import LinearRegression, Lasso

#создадим генератор для разделения данных по индексам
indexes_split = kf.split(x)
#будем записывать качество только на тестовой выборке
losses_test = []

for tr_i, ts_i in indexes_split:
    #получим индексы дата фрейма (id) -=-=-=-=-=-=-=-=-=-=-
    tr_id = x.index[tr_i]
    ts_id = x.index[ts_i]

    #разделим выборку на трейн и тест -=-=-=-=-=-=-=-=-=-=-
    x_train = x.loc[tr_id]
    y_train = y.loc[tr_id]
    #
    x_test = x.loc[ts_id]
    y_test = y.loc[ts_id]

    #обучим модель -=-=-=-=-=-=-=-=-=-=--=-=-=-=-=-=-=-=-=-
    model = LinearRegression()
    model.fit(x_train, y_train)

    #сохраним ошибку модели на тесте -=-=-=-=-=-=-=-=-=-=-
    pred = model.predict(x_test)
    error_for_objects = (pred - y_test)**2
    error_average = error_for_objects.mean()
    losses_test.append(error_average)
print(np.mean(losses_test))

27174867.511216953


_**Замерим качество после добавления линейно зависимых признаков.**_

_Создадим линейно зависимые признаки._

In [25]:
x_bad = x.copy()

for i in range(20):
    x_bad[f'f{i}'] = x_bad['distance_km'] * (i + 1)

_Проведем кросс-валидацию с помощью cross_validate._

In [28]:
from sklearn.model_selection import cross_validate
#обновим генератор разделения данных
kf = KFold(n_splits=4, shuffle=True, random_state=33)
indexes_split = kf.split(x)

model = LinearRegression()
#проведем кросс валидацию
cv_res_1 = cross_validate(model, x_bad, y,
                          scoring='neg_mean_squared_error',
                          cv=kf,
                          return_train_score=True)
cross_val_error = -np.mean(cv_res_1['test_score'])
print('изменение ошибки после добавления новых признаков')
print(cross_val_error - np.mean(losses_test))
cross_val_error


изменение ошибки после добавления новых признаков
0.0


np.float64(27174867.511216953)

_Качество должно ухудшиться. Проверим признаки на мультиколлинеарность._

In [29]:
print('первая модель')
#узнаем ранг матрицы (кол-во линейной независимых столбцов)
rank_x = np.linalg.matrix_rank(x)
print(f'кол-во признаков: {x.shape[1]}')
print(f'кол-во независимых: {rank_x}')

первая модель
кол-во признаков: 4
кол-во независимых: 4


In [30]:
print('вторая модель')
#узнаем ранг матрицы (кол-во линейной независимых столбцов)
rank_x = np.linalg.matrix_rank(x_bad)
print(f'кол-во признаков: {x_bad.shape[1]}')
print(f'кол-во независимых: {rank_x}')

вторая модель
кол-во признаков: 24
кол-во независимых: 4


#### **Применим Лассо-регрессию.**

_Будем применять минимаксное масштабирование._

In [31]:
from sklearn.preprocessing import MinMaxScaler

_Возьмем объект Pipeline._

 _Pipeline — объект, объединяющий несколько последовательных шагов обработки данных и обучения модели._

In [32]:
from sklearn.pipeline import Pipeline

_Для поиска наилучшего параметра влияния регуляризации используем GridSearchCV._

_GridSearchCV — это объект, который перебирает заданные варианты параметров модели, проверяет каждый вариант с помощью кросс-валидации и выбирает лучший._

In [33]:
from sklearn.model_selection import GridSearchCV

_Укажем способ масштабирования и модель._

In [35]:
from sklearn.linear_model import Lasso
pipe_lasso = Pipeline([('scaler', MinMaxScaler()),
                       ('lasso', Lasso(max_iter=100_000))])

_Создадим 30 разных параметров альфа._

In [36]:
alphas = np.linspace(0.000_01, 0.000_1, num=30)

_Начнем поиск параметров._

In [37]:
cv_lasso = GridSearchCV(pipe_lasso,
                        param_grid={'lasso__alpha': alphas},
                        scoring='neg_mean_squared_error',
                        cv=kf,
                        return_train_score=True)
#вызовем обучение
cv_lasso.fit(x_bad, y)

KeyboardInterrupt: 